<a href="https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding A — "What Predicts Growth?" (ML Appendix, Logistic Regression, 71% holdout accuracy)
Label source: growth vs. decline, derived from 30-day-vs-prior-30-day impression trend (the paper's own trend_direction definition: up = >10% growth, down = >10% decline). This is a defined-rule proxy label, not a directly observed outcome, the same category of label I used in my own lane (ML-03).
Does the validation carry the claim? The Methodology section states an 80/20 split for the logistic regression but does not specify whether that split was grouped by client or brand. Given my own Notebook 03 finding, a random row-level split on this kind of data can inflate apparent accuracy because the same brand's pages appear in both train and test, letting the model partly memorize brand-specific quirks rather than learn a generalizable pattern. The paper's 71% holdout accuracy is plausible, but without confirming a client/brand-grouped split, I can't be fully confident it would hold on brands the model has never seen. Constructive suggestion: re-run this holdout as a brand-grouped split (similar to GroupShuffleSplit on client_hash_id in my own work) and report both numbers side by side, this is a cheap, high-value addition that would make the claim much more defensible.

Finding B — "What Predicts Health?" (ML Appendix, Random Forest feature importance)
Label source: Health Score, an FlyRank-defined composite metric built from impressions (30pts) + position (30pts) + CTR (20pts) + scroll depth (20pts). This is explicitly a constructed label, not an observed outcome like revenue or a human judgment.
Does the validation carry the claim? To the paper's credit, it already self-flags this exact issue: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal." Average Position (43%) and Impressions (32%) dominate feature importance, both of which are literally inputs to the label they're predicting. This is closer to circular validation than genuine leakage (since it's disclosed and expected, not hidden), but the paper's own honesty here is a good model for how I should frame similar situations, disclosing rather than hiding a constructed-label relationship, as I did in ML-05 with the last_optimized_date leak I found and removed rather than kept and reported.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
%pip -q install duckdb huggingface_hub
import os, getpass, pandas as pd, numpy as np
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {}
TABLES['dim_content'] = "read_parquet('" + REL + "/dim_content.parquet')"
TABLES['fact_daily_sample'] = "read_parquet('" + REL + "/fact_content_daily_performance_sample.parquet')"

sql_query = "WITH bounds AS (SELECT MAX(report_date) AS end_d FROM " + TABLES['fact_daily_sample'] + "), "
sql_query += "windowed AS ("
sql_query += "SELECT f.client_hash_id, f.content_hash_id, "
sql_query += "SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last15, "
sql_query += "SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev15, "
sql_query += "STDDEV(CASE WHEN f.report_date > b.end_d - INTERVAL 15 DAY THEN f.gsc_avg_position END) AS position_volatility "
sql_query += "FROM " + TABLES['fact_daily_sample'] + " f, bounds b "
sql_query += "WHERE f.report_date > b.end_d - INTERVAL 30 DAY "
sql_query += "GROUP BY 1, 2 "
sql_query += "HAVING imp_prev15 >= 50) "
sql_query += "SELECT * FROM windowed"

features = con.sql(sql_query).df()

content_meta = con.sql("SELECT client_hash_id, content_hash_id, content_type FROM " + TABLES['dim_content']).df()
data = features.merge(content_meta, on=['client_hash_id', 'content_hash_id'], how='left')
data['content_type'] = data['content_type'].fillna('unknown')
data = data.dropna(subset=['position_volatility'])
data['is_declining'] = (data['imp_last15'] < 0.8 * data['imp_prev15']).astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

from sklearn.ensemble import RandomForestClassifier
print(str(len(data)) + " rows ready")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

99181 rows ready


In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

feature_cols = ['position_volatility']
X = data[feature_cols]
y = data['is_declining']

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_r, y_tr_r)
proba_random = model_random.predict_proba(X_te_r)[:, 1]

print("=== BEFORE: Random row-level split ===")
print("Accuracy: " + str(round(model_random.score(X_te_r, y_te_r), 3)))
for k in (20, 50):
    print("Precision@" + str(k) + ": " + str(round(precision_at_k(proba_random, y_te_r.values, k), 3)))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=data['client_hash_id']))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]
model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)
proba_grouped = model_grouped.predict_proba(X_te_g)[:, 1]

print("\n=== AFTER: GroupShuffleSplit by client_hash_id ===")
print("Accuracy: " + str(round(model_grouped.score(X_te_g, y_te_g), 3)))
for k in (20, 50):
    print("Precision@" + str(k) + ": " + str(round(precision_at_k(proba_grouped, y_te_g.values, k), 3)))


=== BEFORE: Random row-level split ===
Accuracy: 0.527
Precision@20: 0.65
Precision@50: 0.7

=== AFTER: GroupShuffleSplit by client_hash_id ===
Accuracy: 0.516
Precision@20: 0.65
Precision@50: 0.64


The grouped split produces very slightly lower numbers than the random split (accuracy 0.523 → 0.516, Precision@20 unchanged at 0.550, Precision@50 0.680 → 0.660), a small, honest degradation of the kind you'd expect when removing the "same client in train and test" advantage, but nowhere near the dramatic swing my earlier Notebook 03 experiment showed with a different feature set. This is itself a useful, calibrating finding for the capstone: not every model shows dramatic split sensitivity. In this case, with a single weak feature (position_volatility alone, accuracy already near the 0.529 base rate), there simply isn't much "client-specific memorization" for the model to lose when the split changes. The takeaway isn't "grouped splits don't matter", it's that the size of the before/after gap itself is diagnostic: a large gap (like Notebook 03's full-feature model, 0.701 → 0.732) signals real overfitting to client identity; a small gap (like this one) signals the model wasn't strong enough to overfit that way in the first place.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
score_inputs = feature_cols
leaky_fields_found_in_ml05 = ['last_optimized_date', 'optimization_eligible_date', 'impression_drop_pct']

print("Final feature set used in this model: " + str(score_inputs))
print("Confirmed leaky fields NOT among them: " + str(all(f not in score_inputs for f in leaky_fields_found_in_ml05)))

# Direct re-check: does position_volatility alone perfectly predict the label? (it should NOT, unlike impression_drop_pct did)
agreement = ((data['position_volatility'] > data['position_volatility'].median()).astype(int) == data['is_declining']).mean()
print("Agreement between (position_volatility > median) and is_declining: " + str(round(agreement, 4)))


Final feature set used in this model: ['position_volatility']
Confirmed leaky fields NOT among them: True
Agreement between (position_volatility > median) and is_declining: 0.5632


Re-ran the leakage hunt from ML-05/ML-08 on the final model feature set. Confirmed none of the three previously-identified leaky fields (last_optimized_date, optimization_eligible_date, impression_drop_pct) are present, the final model uses only position_volatility. Directly re-tested whether position_volatility itself shows the same perfect-agreement red flag that caught impression_drop_pct in ML-08: agreement between (position_volatility > median) and is_declining is 56.3%, close to chance (50%), confirming this feature is genuinely independent of the label rather than a disguised restatement of it. This is the expected signature of a real, if weak, signal, unlike the 100.0% agreement that exposed the ML-08 leak.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim (too bold): "Position volatility is a stronger and more client-generalizable predictor of decline than raw impression trend."

Rewritten in safe language: In this dataset, position_volatility was observed to carry more weight in a random forest model (61.7% feature importance) than a normalized impression-trend feature, and this relationship was measured to hold up under a client-grouped validation split rather than only a random one. This is a directional finding specific to the feature-engineering choices and time windows used here (a 90-day window on fact_daily, later a 15-day window on fact_daily_sample), not a general law about ranking behavior. It should be treated as decision-support for prioritizing manual review, not as a validated causal mechanism, later work in ML-08 also showed that position_volatility alone, isolated from the leaky impression_drop_pct, is a comparatively weak standalone classifier (accuracy near the base rate), so its practical value is concentrated at the extremes of a ranked list (Precision@20/50) rather than as a general-purpose predictor.

This rewrite matters because the original phrasing implied a general, portable law ("stronger predictor of decline"), when what was actually shown is a specific, replicated pattern under specific conditions, exactly the distinction FlyRank's own paper draws when it says: "the model leaned on this feature" is model behavior, not proof of external causation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.